In [ ]:
"""
Sistema de recomendação de cursos baseado no perfil profissional.

Entrada:
  - Área de atuação
  - Prioridade (carreira, atualização, transição)
  - Estilo de estudo (rápido, profundo, equilibrado)
  - Horas semanais disponíveis
  - Orçamento máximo

Saída:
  - Catálogo personalizado de cursos com impacto ajustado
  - Catálogo ordenado (merge sort) por impacto ajustado
  - Plano ótimo de estudos usando a ideia da mochila (knapsack)
  - Relatório de horas, orçamento e relevância total

Objetivo:
  Maximizar a relevância (impacto ajustado) dos cursos escolhidos,
  respeitando as restrições de horas e orçamento.
"""

from functools import lru_cache
from IPython.display import display
import pandas as pd

# ============================================================
# 1) Função de saída formatada (recursão + memoização)
# ============================================================

def mostrar(texto):
    """
    Mostra o texto no console linha a linha (print),
    usando recursão e memoização nas linhas.
    """
    linhas = texto.split("\n")

    @lru_cache(maxsize=None)
    def get_linha(i):
        if i < 0 or i >= len(linhas):
            return ""
        return linhas[i]

    def imprimir(i=0):
        if i == len(linhas):
            return
        print(get_linha(i))
        return imprimir(i + 1)

    imprimir(0)


# ============================================================
# 2) Catálogo base com pelo menos 20 cursos
# ============================================================

CURSOS_BASE = {
    "saúde": [
        {"curso": "IA em Análise de Exames de Imagem", "area": "Saúde", "horas": 8, "preco": 900, "impacto": 9},
        {"curso": "Prontuário Eletrônico e Assistentes de IA", "area": "Saúde", "horas": 6, "preco": 700, "impacto": 8},
        {"curso": "Telemedicina e Triagem Automatizada", "area": "Saúde", "horas": 10, "preco": 1100, "impacto": 9},
        {"curso": "Gestão de Clínicas com IA", "area": "Saúde", "horas": 5, "preco": 600, "impacto": 7},
        {"curso": "IA para Enfermagem Baseada em Evidências", "area": "Saúde", "horas": 7, "preco": 800, "impacto": 8},
        {"curso": "Monitoramento de Pacientes com Sensores e IA", "area": "Saúde", "horas": 9, "preco": 950, "impacto": 9},
    ],
    "tecnologia": [
        {"curso": "Fundamentos de IA Generativa para Devs", "area": "Tecnologia", "horas": 6, "preco": 650, "impacto": 8},
        {"curso": "Integração de APIs de IA em Sistemas", "area": "Tecnologia", "horas": 10, "preco": 1200, "impacto": 9},
        {"curso": "Automação de Testes com IA", "area": "Tecnologia", "horas": 7, "preco": 750, "impacto": 8},
        {"curso": "Chatbots e Assistentes Virtuais", "area": "Tecnologia", "horas": 8, "preco": 900, "impacto": 9},
        {"curso": "MLOps e Deploy de Modelos em Produção", "area": "Tecnologia", "horas": 12, "preco": 1500, "impacto": 10},
        {"curso": "Segurança da Informação com IA", "area": "Tecnologia", "horas": 9, "preco": 1100, "impacto": 9},
    ],
    "gestão": [
        {"curso": "Tomada de Decisão Orientada a Dados", "area": "Gestão", "horas": 6, "preco": 700, "impacto": 8},
        {"curso": "Planejamento Estratégico com IA", "area": "Gestão", "horas": 8, "preco": 900, "impacto": 9},
        {"curso": "Previsão de Demanda e Vendas", "area": "Gestão", "horas": 10, "preco": 1100, "impacto": 9},
        {"curso": "Gestão de Projetos com Ferramentas de IA", "area": "Gestão", "horas": 7, "preco": 800, "impacto": 8},
        {"curso": "Análise de Risco e Cenários com IA", "area": "Gestão", "horas": 9, "preco": 1000, "impacto": 9},
        {"curso": "People Analytics para RH", "area": "Gestão", "horas": 6, "preco": 750, "impacto": 8},
    ],
    "educação": [
        {"curso": "Planejamento de Aulas com IA", "area": "Educação", "horas": 6, "preco": 600, "impacto": 8},
        {"curso": "Correção Automatizada de Avaliações", "area": "Educação", "horas": 7, "preco": 700, "impacto": 8},
        {"curso": "Personalização de Trilhas de Aprendizagem", "area": "Educação", "horas": 9, "preco": 900, "impacto": 9},
        {"curso": "Análise de Engajamento em Plataformas EAD", "area": "Educação", "horas": 8, "preco": 850, "impacto": 8},
        {"curso": "Gamificação com Suporte de IA", "area": "Educação", "horas": 6, "preco": 650, "impacto": 8},
        {"curso": "Inclusão e Acessibilidade com IA na Educação", "area": "Educação", "horas": 7, "preco": 750, "impacto": 9},
    ],
}

# ============================================================
# 3) Montar lista de cursos (recursão + memoização)
# ============================================================

def montar_lista_cursos(area_foco):
    """
    Retorna uma lista de cursos de acordo com a área de foco escolhida.
    Usa recursão para concatenar listas de áreas e memoização para
    armazenar os cursos de cada área.
    """
    area_foco = area_foco.strip().lower()

    @lru_cache(maxsize=None)
    def cursos_por_area(area):
        area = area.lower()
        return tuple(CURSOS_BASE.get(area, []))

    if area_foco == "todas":
        areas = list(CURSOS_BASE.keys())
    else:
        areas = [area_foco]

    def juntar_areas(idx=0):
        if idx == len(areas):
            return []
        area = areas[idx]
        cursos_area = list(cursos_por_area(area))
        return cursos_area + juntar_areas(idx + 1)

    return juntar_areas(0)


# ============================================================
# 4) Coletar perfil do usuário (recursão + memoização)
# ============================================================

def coletar_perfil_usuario():
    """
    Coleta o perfil do usuário de forma recursiva (perguntas em sequência).
    Usa um dicionário de memoização para armazenar respostas intermediárias.
    """
    perguntas = [
        ("area", "Informe sua área de atuação (saúde, tecnologia, gestão, educação ou todas): "),
        ("prioridade", "Qual é sua prioridade principal? (carreira, atualização, transição): "),
        ("estilo", "Qual é seu estilo de estudo? (rápido, profundo, equilibrado): "),
        ("horas_max", "Quantas horas semanais você pode estudar? "),
        ("orcamento_max", "Qual é seu orçamento máximo (em R$)? "),
    ]

    memo_respostas = {}

    def perguntar(i=0, respostas=None):
        if respostas is None:
            respostas = {}

        if i == len(perguntas):
            return respostas

        if i in memo_respostas:
            respostas.update(memo_respostas[i])
            return perguntar(i + 1, respostas)

        chave, texto = perguntas[i]
        valor = input(texto)

        if chave in ("horas_max", "orcamento_max"):
            try:
                valor = float(valor)
            except ValueError:
                mostrar("Valor inválido. Tente novamente.")
                return perguntar(i, respostas)

        respostas[chave] = valor
        memo_respostas[i] = respostas.copy()
        return perguntar(i + 1, respostas)

    return perguntar()


# ============================================================
# 5) Calcular relevância ajustada (recursão + memoização, impacto máx 10)
# ============================================================

def calcular_relevancia_cursos(cursos, area_foco, prioridade, estilo):
    """
    Ajusta o impacto dos cursos de acordo com a prioridade, área de foco
    e estilo de estudo. Usa recursão para percorrer a lista de cursos
    e memoização para os pesos de prioridade/estilo.

    O impacto_ajustado é limitado a, no máximo, 10.
    """
    area_foco = area_foco.strip().lower()
    prioridade = prioridade.strip().lower()
    estilo = estilo.strip().lower()

    @lru_cache(maxsize=None)
    def peso_prioridade(p):
        mapa = {
            "carreira": 1.2,
            "atualização": 1.1,
            "atualizacao": 1.1,
            "transição": 1.3,
            "transicao": 1.3,
        }
        return mapa.get(p, 1.0)

    @lru_cache(maxsize=None)
    def peso_estilo(e):
        mapa = {
            "rápido": 1.1,
            "rapido": 1.1,
            "profundo": 1.2,
            "equilibrado": 1.0,
        }
        return mapa.get(e, 1.0)

    def processar_indice(i):
        if i == len(cursos):
            return []
        curso = dict(cursos[i])
        impacto_base = curso["impacto"]

        bonus_area = 1.1 if area_foco != "todas" and curso["area"].lower() == area_foco else 1.0
        impacto_ajustado = impacto_base * peso_prioridade(prioridade) * peso_estilo(estilo) * bonus_area

        if impacto_ajustado > 10:
            impacto_ajustado = 10

        curso["impacto_ajustado"] = round(impacto_ajustado, 2)

        return [curso] + processar_indice(i + 1)

    return processar_indice(0)


# ============================================================
# 6) Merge sort recursivo com memoização
# ============================================================

def merge_sort_lista(lista, chave):
    """
    Ordena uma lista de dicionários usando merge sort recursivo.
    Usa memoização em subintervalos (start, end).
    Retorna uma nova lista ordenada de forma decrescente pela chave.
    """

    @lru_cache(maxsize=None)
    def sort_interval(start, end):
        if end - start <= 1:
            return tuple(lista[start:end])

        mid = (start + end) // 2
        left = sort_interval(start, mid)
        right = sort_interval(mid, end)

        i = j = 0
        merged = []
        while i < len(left) and j < len(right):
            if left[i][chave] >= right[j][chave]:
                merged.append(left[i])
                i += 1
            else:
                merged.append(right[j])
                j += 1
        while i < len(left):
            merged.append(left[i])
            i += 1
        while j < len(right):
            merged.append(right[j])
            j += 1

        return tuple(merged)

    if not lista:
        return []

    ordenada = sort_interval(0, len(lista))
    return list(ordenada)


# ============================================================
# 7) Mochila (knapsack) com recursão + memoização
# ============================================================

def montar_plano_otimo(cursos, horas_max, orcamento_max):
    """
    Aplica a ideia da mochila 0/1 para escolher o melhor conjunto de cursos,
    maximizando o impacto_ajustado dentro das restrições de horas e orçamento.
    Usa recursão + memoização (lru_cache).
    Retorna (lista_cursos_escolhidos, horas_usadas, gasto_total, impacto_total).
    """
    n = len(cursos)

    @lru_cache(maxsize=None)
    def knapsack(i, horas_restantes, orcamento_restante):
        if i == n or horas_restantes <= 0 or orcamento_restante <= 0:
            return 0.0

        curso = cursos[i]
        h = curso["horas"]
        p = curso["preco"]
        v = curso["impacto_ajustado"]

        melhor_sem = knapsack(i + 1, horas_restantes, orcamento_restante)

        melhor_com = 0.0
        if h <= horas_restantes and p <= orcamento_restante:
            melhor_com = v + knapsack(i + 1, horas_restantes - h, orcamento_restante - p)

        return max(melhor_sem, melhor_com)

    def reconstruir(i, horas_restantes, orcamento_restante):
        if i == n or horas_restantes <= 0 or orcamento_restante <= 0:
            return []

        curso = cursos[i]
        h = curso["horas"]
        p = curso["preco"]
        v = curso["impacto_ajustado"]

        valor_atual = knapsack(i, horas_restantes, orcamento_restante)
        valor_sem = knapsack(i + 1, horas_restantes, orcamento_restante)

        if h <= horas_restantes and p <= orcamento_restante:
            valor_com = v + knapsack(i + 1, horas_restantes - h, orcamento_restante - p)
            if valor_com >= valor_sem and valor_com == valor_atual:
                return [curso] + reconstruir(i + 1, horas_restantes - h, orcamento_restante - p)

        return reconstruir(i + 1, horas_restantes, orcamento_restante)

    impacto_max = knapsack(0, int(horas_max), int(orcamento_max))
    cursos_escolhidos = reconstruir(0, int(horas_max), int(orcamento_max))

    horas_usadas = sum(c["horas"] for c in cursos_escolhidos)
    gasto_total = sum(c["preco"] for c in cursos_escolhidos)

    return cursos_escolhidos, horas_usadas, gasto_total, impacto_max


# ============================================================
# 8) Função principal com loop recursivo + memo (cache simples)
# ============================================================

def executar_sistema():
    """
    Função principal que orquestra a solução:
    - coleta perfil
    - monta catálogo
    - aplica ordenação (merge sort)
    - aplica mochila
    Usa um loop recursivo e um dicionário como cache (memoização)
    para evitar recomputar catálogo e plano ótimo.
    """
    cache = {
        "perfil": None,
        "catalogo": None,
        "catalogo_ordenado": None,
        "plano": None,
    }

    def loop():
        menu = (
            "================ MENU =================\n"
            "1 - Informar/atualizar perfil profissional\n"
            "2 - Gerar catálogo personalizado de cursos\n"
            "3 - Calcular plano ótimo (mochila)\n"
            "0 - Sair\n"
            "=======================================\n"
        )
        mostrar(menu)
        opcao = input("Escolha uma opção: ").strip()

        if opcao == "1":
            perfil = coletar_perfil_usuario()
            cache["perfil"] = perfil
            cache["catalogo"] = None
            cache["catalogo_ordenado"] = None
            cache["plano"] = None
            mostrar("Perfil atualizado com sucesso!")
            return loop()

        elif opcao == "2":
            if cache["perfil"] is None:
                mostrar("Você precisa informar o perfil primeiro (opção 1).")
                return loop()

            perfil = cache["perfil"]
            cursos = montar_lista_cursos(perfil["area"])
            cursos_relevantes = calcular_relevancia_cursos(
                cursos,
                perfil["area"],
                perfil["prioridade"],
                perfil["estilo"],
            )

            cache["catalogo"] = cursos_relevantes

            df = pd.DataFrame(cursos_relevantes)
            mostrar("Catálogo personalizado (antes da ordenação):")
            display(df)

            cursos_ordenados = merge_sort_lista(cursos_relevantes, "impacto_ajustado")
            cache["catalogo_ordenado"] = cursos_ordenados

            df_ord = pd.DataFrame(cursos_ordenados)
            mostrar("Catálogo ordenado por impacto ajustado (decrescente):")
            display(df_ord)

            return loop()

        elif opcao == "3":
            if cache["perfil"] is None or cache["catalogo_ordenado"] is None:
                mostrar("Você precisa gerar o catálogo primeiro (opção 2).")
                return loop()

            perfil = cache["perfil"]
            cursos_ordenados = cache["catalogo_ordenado"]

            plano, horas_usadas, gasto_total, impacto_max = montar_plano_otimo(
                cursos_ordenados,
                perfil["horas_max"],
                perfil["orcamento_max"],
            )

            cache["plano"] = plano

            df_plano = pd.DataFrame(plano)
            mostrar("Plano ótimo de estudos (solução da mochila):")
            display(df_plano)

            resumo = (
                f"Horas usadas: {horas_usadas}\n"
                f"Orçamento gasto: R$ {gasto_total:.2f}\n"
                f"Relevância total (impacto ajustado): {impacto_max:.2f}\n"
            )
            mostrar(resumo)

            return loop()

        elif opcao == "0":
            mostrar("Encerrando o sistema. Obrigado por utilizar a plataforma!")
            return

        else:
            mostrar("Opção inválida, tente novamente.")
            return loop()

    loop()


executar_sistema()


================ MENU =================
1 - Informar/atualizar perfil profissional
2 - Gerar catálogo personalizado de cursos
3 - Calcular plano ótimo (mochila)
0 - Sair

Perfil atualizado com sucesso!
================ MENU =================
1 - Informar/atualizar perfil profissional
2 - Gerar catálogo personalizado de cursos
3 - Calcular plano ótimo (mochila)
0 - Sair

Catálogo personalizado (antes da ordenação):


,curso,area,horas,preco,impacto,impacto_ajustado
0,Tomada de Decisão Orientada a Dados,Gestão,6,700,8,10
1,Planejamento Estratégico com IA,Gestão,8,900,9,10
2,Previsão de Demanda e Vendas,Gestão,10,1100,9,10
3,Gestão de Projetos com Ferramentas de IA,Gestão,7,800,8,10
4,Análise de Risco e Cenários com IA,Gestão,9,1000,9,10
5,People Analytics para RH,Gestão,6,750,8,10


Catálogo ordenado por impacto ajustado (decrescente):


,curso,area,horas,preco,impacto,impacto_ajustado
0,Tomada de Decisão Orientada a Dados,Gestão,6,700,8,10
1,Planejamento Estratégico com IA,Gestão,8,900,9,10
2,Previsão de Demanda e Vendas,Gestão,10,1100,9,10
3,Gestão de Projetos com Ferramentas de IA,Gestão,7,800,8,10
4,Análise de Risco e Cenários com IA,Gestão,9,1000,9,10
5,People Analytics para RH,Gestão,6,750,8,10


================ MENU =================
1 - Informar/atualizar perfil profissional
2 - Gerar catálogo personalizado de cursos
3 - Calcular plano ótimo (mochila)
0 - Sair

Plano ótimo de estudos (solução da mochila):


,curso,area,horas,preco,impacto,impacto_ajustado
0,Tomada de Decisão Orientada a Dados,Gestão,6,700,8,10


Horas usadas: 6
Orçamento gasto: R$ 700.00
Relevância total (impacto ajustado): 10.00

================ MENU =================
1 - Informar/atualizar perfil profissional
2 - Gerar catálogo personalizado de cursos
3 - Calcular plano ótimo (mochila)
0 - Sair

Encerrando o sistema. Obrigado por utilizar a plataforma!
